<a href="https://colab.research.google.com/github/harsha-9977/AIML/blob/main/model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pybullet --quiet

import pybullet as p
import pybullet_data
import time

# Connect to GUI
p.connect(p.DIRECT)

# Set path and gravity
p.setAdditionalSearchPath(pybullet_data.getDataPath())
p.setGravity(0, 0, -9.8)

# Load plane and table
plane_id = p.loadURDF("plane.urdf")
table_id = p.loadURDF("table/table.urdf", basePosition=[0.5, 0, 0])

# Load Franka Panda robot
panda_id = p.loadURDF("franka_panda/panda.urdf", basePosition=[0, 0, 0])

# Load a cube to interact with
cube_id = p.loadURDF("cube_small.urdf", basePosition=[0.6, 0, 0.62])

# Let simulation run
for _ in range(240):
    p.stepSimulation()
    time.sleep(1./240.)


In [2]:
import numpy as np
from PIL import Image

# Get cube position and orientation
cube_pos, cube_orn = p.getBasePositionAndOrientation(cube_id)
print("🟢 Cube 3D Position:", cube_pos)

# Setup virtual camera
view_matrix = p.computeViewMatrix(
    cameraEyePosition=[1, 0, 1],      # Camera position
    cameraTargetPosition=[0.5, 0, 0.6],  # Where it looks
    cameraUpVector=[0, 0, 1]
)

projection_matrix = p.computeProjectionMatrixFOV(
    fov=60, aspect=1.0, nearVal=0.1, farVal=2.0
)

# Capture image
width, height, rgb_img, _, _ = p.getCameraImage(
    width=224, height=224,
    viewMatrix=view_matrix,
    projectionMatrix=projection_matrix,
    renderer=p.ER_BULLET_HARDWARE_OPENGL
)

# Save RGB image
rgb_array = np.reshape(rgb_img, (224, 224, 4))[:, :, :3]
img = Image.fromarray(rgb_array.astype(np.uint8))
img.save("simulated_rgb.png")
print("✅ Saved image as simulated_rgb.png")


🟢 Cube 3D Position: (0.6000010470328385, 8.322969137112462e-06, 0.6499880838171597)
✅ Saved image as simulated_rgb.png


In [3]:
# Define grasp pose (same as cube center + offset above)
grasp_position = list(cube_pos)  # Copy exact object position

# You could add a small offset in Z if needed for grasping above surface
grasp_position[2] += 0.03  # Raise gripper slightly above cube

# Log gripper state
gripper_state = "closed"

# Save all as JSON ground truth
import json

ground_truth = {
    "prompt": "pick the cube from the table",
    "object_label": "cube",
    "object_position": cube_pos,
    "grasp_position": grasp_position,
    "gripper_state": gripper_state,
    "image_file": "simulated_rgb.png"
}

with open("ground_truth.json", "w") as f:
    json.dump(ground_truth, f, indent=2)

print("✅ Ground truth saved to ground_truth.json")


✅ Ground truth saved to ground_truth.json


In [4]:
import os
import random

# Directory to store data
os.makedirs("dataset", exist_ok=True)

samples = []

for i in range(5):
    # Random x, y near center of table
    x = random.uniform(0.55, 0.65)
    y = random.uniform(-0.05, 0.05)
    z = 0.62  # top of table

    # Move cube to new position
    p.resetBasePositionAndOrientation(cube_id, [x, y, z], [0, 0, 0, 1])

    # Let it settle
    for _ in range(30):
        p.stepSimulation()

    # Render camera
    _, _, rgb_img, _, _ = p.getCameraImage(
        width=224, height=224,
        viewMatrix=view_matrix,
        projectionMatrix=projection_matrix,
        renderer=p.ER_BULLET_HARDWARE_OPENGL
    )
    rgb_array = np.reshape(rgb_img, (224, 224, 4))[:, :, :3]

    # Save image
    img_file = f"dataset/rgb_{i:03d}.png"
    Image.fromarray(rgb_array.astype(np.uint8)).save(img_file)

    # Get object pos
    obj_pos, _ = p.getBasePositionAndOrientation(cube_id)

    # Define grasp pos
    grasp_pos = list(obj_pos)
    grasp_pos[2] += 0.03

    # Save ground truth JSON
    gt = {
        "prompt": "pick the cube from the table",
        "object_label": "cube",
        "object_position": obj_pos,
        "grasp_position": grasp_pos,
        "gripper_state": "closed",
        "image_file": img_file
    }

    with open(f"dataset/gt_{i:03d}.json", "w") as f:
        json.dump(gt, f, indent=2)

    samples.append(gt)

print(f"✅ Generated {len(samples)} ground truth samples.")


✅ Generated 5 ground truth samples.


In [12]:
import torch
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image
import glob

# Image transform
img_transform = transforms.Compose([
    transforms.ToTensor(),  # [0,1] CxHxW
])

class GraspDataset(Dataset):
    def __init__(self, dataset_dir):
        self.image_paths = sorted(glob.glob(f"{dataset_dir}/rgb_*.png"))
        self.json_paths = sorted(glob.glob(f"{dataset_dir}/gt_*.json"))

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx])
        image_tensor = img_transform(image)

        with open(self.json_paths[idx], 'r') as f:
            data = json.load(f)

        prompt = data["prompt"]
        object_pos = torch.tensor(data["object_position"], dtype=torch.float32)
        grasp_pos = torch.tensor(data["grasp_position"], dtype=torch.float32)
        prompt_tokens = tokenize_prompt(prompt)

        return {
            "image": image_tensor,
            "prompt_tokens": prompt_tokens,
            "grasp_position": grasp_pos,
            "prompt": prompt  # 🟢 Add this back for debugging
        }



# Load and test
dataset = GraspDataset("dataset")
sample = dataset[0]
print("🧪 Sample loaded:")
print("Prompt:", sample["prompt"])
print("Image shape:", sample["image"].shape)
print("Grasp Position:", sample["grasp_position"])


🧪 Sample loaded:
Prompt: pick the cube from the table
Image shape: torch.Size([3, 224, 224])
Grasp Position: tensor([ 0.6016, -0.0327,  0.6775])


In [13]:
import torch.nn as nn
import torchvision.models as models

class RT1Mini(nn.Module):
    def __init__(self, output_dim=3):
        super().__init__()

        # Vision encoder (pretrained ResNet18)
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.vision_encoder = nn.Sequential(*list(resnet.children())[:-1])  # remove final FC

        # Text encoder (embedding + LSTM)
        self.embedding = nn.Embedding(1000, 32)  # small vocab
        self.lstm = nn.LSTM(input_size=32, hidden_size=64, batch_first=True)

        # Final regressor
        self.fc = nn.Sequential(
            nn.Linear(512 + 64, 256),
            nn.ReLU(),
            nn.Linear(256, output_dim)  # x, y, z
        )

    def forward(self, image, prompt_tokens):
        # Vision encoding
        x_img = self.vision_encoder(image).squeeze(-1).squeeze(-1)  # [B, 512]

        # Prompt encoding
        x_txt, _ = self.lstm(self.embedding(prompt_tokens))  # [B, seq_len, 64]
        x_txt = x_txt[:, -1, :]  # take last token output

        # Combine
        x = torch.cat([x_img, x_txt], dim=1)

        # Predict grasp position
        return self.fc(x)


In [14]:
from torch.nn.utils.rnn import pad_sequence

# Define small vocabulary
vocab = {
    "pick": 1,
    "the": 2,
    "cube": 3,
    "from": 4,
    "table": 5,
    "<pad>": 0
}

def tokenize_prompt(prompt, max_len=6):
    words = prompt.lower().split()
    tokens = [vocab.get(w, 0) for w in words]
    tokens = tokens[:max_len]
    tokens += [vocab["<pad>"]] * (max_len - len(tokens))
    return torch.tensor(tokens)


In [15]:
def __getitem__(self, idx):
    image = Image.open(self.image_paths[idx])
    image_tensor = img_transform(image)

    with open(self.json_paths[idx], 'r') as f:
        data = json.load(f)

    prompt = data["prompt"]
    object_pos = torch.tensor(data["object_position"], dtype=torch.float32)
    grasp_pos = torch.tensor(data["grasp_position"], dtype=torch.float32)
    prompt_tokens = tokenize_prompt(prompt)

    return {
        "image": image_tensor,
        "prompt_tokens": prompt_tokens,
        "grasp_position": grasp_pos
    }


In [16]:
from torch.utils.data import DataLoader

model = RT1Mini()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.MSELoss()

def custom_collate(batch):
    images = torch.stack([b["image"] for b in batch])
    tokens = torch.stack([b["prompt_tokens"] for b in batch])  # [B, seq_len]
    targets = torch.stack([b["grasp_position"] for b in batch])
    return {
        "image": images,
        "prompt_tokens": tokens,
        "grasp_position": targets
    }


# DataLoader
loader = DataLoader(dataset, batch_size=2, shuffle=True, collate_fn=custom_collate)

# Train 10 epochs
for epoch in range(10):
    total_loss = 0.0
    for batch in loader:
        img = batch["image"]
        tokens = batch["prompt_tokens"]
        target = batch["grasp_position"]

        # Forward
        pred = model(img, tokens)

        # Loss
        loss = criterion(pred, target)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}: Loss = {total_loss:.4f}")


Epoch 1: Loss = 1.0537
Epoch 2: Loss = 0.1160
Epoch 3: Loss = 0.0301
Epoch 4: Loss = 0.0494
Epoch 5: Loss = 0.0172
Epoch 6: Loss = 0.0023
Epoch 7: Loss = 0.0013
Epoch 8: Loss = 0.0010
Epoch 9: Loss = 0.0007
Epoch 10: Loss = 0.0009


In [17]:
# Load image and prompt from sample (or new one)
test_idx = 0
sample = dataset[test_idx]

image = sample["image"].unsqueeze(0)         # Add batch dim
tokens = sample["prompt_tokens"].unsqueeze(0)

# Predict
model.eval()
with torch.no_grad():
    pred = model(image, tokens)

print("🧠 Prompt:", sample["prompt"])
print("📍 Predicted Grasp Position:", pred[0].numpy())
print("🎯 Ground Truth Position   :", sample["grasp_position"].numpy())


🧠 Prompt: pick the cube from the table
📍 Predicted Grasp Position: [ 0.5409589  -0.12014717  0.8558721 ]
🎯 Ground Truth Position   : [ 0.60162103 -0.0327451   0.67753917]
